[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

# Model Validation — Apply

Hands-on exercises for ONNX model validation.  You will use `onnx.checker`, run numerical
comparisons between ONNX Runtime and reference implementations, compute error metrics,
build a CI-ready validation harness, and test model equivalence before/after optimization.

## Table of Contents

| # | Exercise | Objective |
|---|----------|----------|
| 1 | [Setup](#1) | Install and import dependencies |
| 2 | [Structural Validation](#2) | Use onnx.checker on valid/invalid models |
| 3 | [Catch Common Errors](#3) | Undefined inputs, bad attributes |
| 4 | [Shape Inference as Validation](#4) | Strict mode type checking |
| 5 | [Error Metrics Implementation](#5) | Code cosine sim, SNR, relative error |
| 6 | [Numerical Validation](#6) | Compare ORT output to NumPy reference |
| 7 | [Error Distribution Analysis](#7) | Matplotlib histograms and plots |
| 8 | [Validation Harness](#8) | Build a reusable test function |
| 9 | [Before/After Optimization](#9) | Test equivalence after graph rewrites |
| 10 | [Edge Case Testing](#10) | Zeros, large values, special inputs |

<a id="1"></a>
## Exercise 1 — Setup

In [ ]:
# !pip install onnx onnxruntime numpy matplotlib --quiet

import numpy as np
import onnx
from onnx import helper, TensorProto, checker, shape_inference

try:
    import onnxruntime as ort
    print(f"ONNX: {onnx.__version__}, ORT: {ort.__version__}")
except ImportError:
    print(f"ONNX: {onnx.__version__} (onnxruntime not installed)")

<a id="2"></a>
## Exercise 2 — Structural Validation with onnx.checker

Build valid and invalid models, run `checker.check_model()`, and observe the errors.

In [ ]:
def build_linear_model(in_features, out_features):
    """Build a simple linear model: Y = X @ W + B."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [None, in_features])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

    W_np = np.random.randn(in_features, out_features).astype(np.float32)
    B_np = np.random.randn(out_features).astype(np.float32)

    W = helper.make_tensor("W", TensorProto.FLOAT, [in_features, out_features], W_np.flatten())
    B = helper.make_tensor("B", TensorProto.FLOAT, [out_features], B_np.flatten())

    graph = helper.make_graph(
        [
            helper.make_node("MatMul", ["X", "W"], ["XW"]),
            helper.make_node("Add", ["XW", "B"], ["Y"]),
        ],
        "linear", [X], [Y], initializer=[W, B],
    )
    return helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)]), W_np, B_np

# Valid model
np.random.seed(42)
model, W_np, B_np = build_linear_model(784, 10)
try:
    checker.check_model(model)
    print("Valid model: PASSED checker")
except Exception as e:
    print(f"Valid model: FAILED — {e}")

# Invalid model: missing input reference
bad_graph = helper.make_graph(
    [helper.make_node("Add", ["X", "MISSING"], ["Y"])],
    "bad",
    [helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])],
    [helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)],
)
bad_model = helper.make_model(bad_graph, opset_imports=[helper.make_opsetid("", 17)])
try:
    checker.check_model(bad_model)
    print("Invalid model: PASSED (unexpected)")
except Exception as e:
    print(f"Invalid model: CAUGHT — {type(e).__name__}")

<a id="3"></a>
## Exercise 3 — Catch Common Validation Errors

Create models with specific types of errors and see what the checker reports.

In [ ]:
errors_found = []

# Error 1: Wrong attribute type for an operator
try:
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
    # Softmax axis should be an int, but the model structure is valid
    softmax = helper.make_node("Softmax", ["X"], ["Y"], axis=-1)
    graph = helper.make_graph([softmax], "softmax_test", [X], [Y])
    model_sm = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    checker.check_model(model_sm)
    print("Softmax with axis=-1: PASSED")
except Exception as e:
    errors_found.append(str(e))
    print(f"Softmax error: {e}")

# Error 2: Initializer shape mismatch
try:
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
    # Declare dims [10, 5] but provide data for [10, 3]
    W_wrong = onnx.TensorProto()
    W_wrong.name = "W"
    W_wrong.data_type = TensorProto.FLOAT
    W_wrong.dims.extend([10, 5])
    W_wrong.float_data.extend(np.zeros(30, dtype=np.float32))  # 30 != 10*5=50

    graph = helper.make_graph(
        [helper.make_node("MatMul", ["X", "W"], ["Y"])],
        "shape_mismatch", [X], [Y], initializer=[W_wrong],
    )
    model_bad = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    checker.check_model(model_bad)
    print("Initializer mismatch: PASSED (unexpected)")
except Exception as e:
    errors_found.append(str(e)[:100])
    print(f"Initializer mismatch: CAUGHT — {type(e).__name__}")

# Error 3: Missing opset for custom domain
try:
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 10])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
    custom = helper.make_node("MyCustomOp", ["X"], ["Y"], domain="custom.domain")
    graph = helper.make_graph([custom], "no_domain", [X], [Y])
    model_nd = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    checker.check_model(model_nd)
    print("Missing domain opset: PASSED (unexpected)")
except Exception as e:
    errors_found.append(str(e)[:100])
    print(f"Missing domain opset: CAUGHT — {type(e).__name__}")

print(f"\nTotal errors caught: {len(errors_found)}")

<a id="4"></a>
## Exercise 4 — Shape Inference as Validation

Use `infer_shapes()` with strict mode to catch type and shape inconsistencies.

In [ ]:
# Build a valid model and run strict shape inference
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [8, 100])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

W1 = helper.make_tensor("W1", TensorProto.FLOAT, [100, 50],
                        np.zeros((100, 50), dtype=np.float32).flatten())
W2 = helper.make_tensor("W2", TensorProto.FLOAT, [50, 10],
                        np.zeros((50, 10), dtype=np.float32).flatten())

graph = helper.make_graph(
    [
        helper.make_node("MatMul", ["X", "W1"], ["H"]),
        helper.make_node("Relu", ["H"], ["R"]),
        helper.make_node("MatMul", ["R", "W2"], ["Y"]),
    ],
    "mlp", [X], [Y], initializer=[W1, W2],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

try:
    inferred = shape_inference.infer_shapes(model, check_type=True)
    print("Shape inference (strict): PASSED")
    for vi in list(inferred.graph.value_info) + list(inferred.graph.output):
        tt = vi.type.tensor_type
        dtype = TensorProto.DataType.Name(tt.elem_type)
        dims = [d.dim_value for d in tt.shape.dim]
        print(f"  {vi.name}: {dtype}{dims}")
except Exception as e:
    print(f"Shape inference error: {e}")

<a id="5"></a>
## Exercise 5 — Implement Error Metrics

Implement the key numerical comparison metrics from scratch:
- Absolute/relative error
- Cosine similarity: $\cos(\theta) = \frac{\mathbf{y} \cdot \hat{\mathbf{y}}}{\|\mathbf{y}\| \|\hat{\mathbf{y}}\|}$
- SNR: $\text{SNR} = 10 \log_{10} \frac{\|\mathbf{y}\|^2}{\|\mathbf{y} - \hat{\mathbf{y}}\|^2}$ dB

In [ ]:
def absolute_error(y_ref, y_test):
    """Element-wise absolute error."""
    return np.abs(y_ref.astype(np.float64) - y_test.astype(np.float64))

def relative_error(y_ref, y_test, delta=1e-8):
    """Element-wise relative error with stability constant."""
    return np.abs(y_ref - y_test).astype(np.float64) / (np.abs(y_ref).astype(np.float64) + delta)

def cosine_similarity(y_ref, y_test):
    """Cosine similarity between flattened vectors."""
    a = y_ref.flatten().astype(np.float64)
    b = y_test.flatten().astype(np.float64)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

def snr_db(y_ref, y_test):
    """Signal-to-noise ratio in decibels."""
    a = y_ref.flatten().astype(np.float64)
    b = y_test.flatten().astype(np.float64)
    signal = np.sum(a ** 2)
    noise = np.sum((a - b) ** 2)
    return 10 * np.log10(signal / (noise + 1e-12))

def allclose_check(y_ref, y_test, atol=1e-5, rtol=1e-4):
    """Check |y_ref - y_test| <= atol + rtol * |y_ref| for all elements."""
    return np.all(np.abs(y_ref - y_test) <= atol + rtol * np.abs(y_ref))

# Demonstrate with controlled noise
np.random.seed(42)
y_ref = np.random.randn(100).astype(np.float32)
noise_levels = [0, 1e-7, 1e-5, 1e-3]

print(f"{'Noise':>10s}  {'Max Abs':>10s}  {'Max Rel':>10s}  {'Cosine':>12s}  {'SNR (dB)':>10s}  {'allclose':>8s}")
print("-" * 70)
for noise in noise_levels:
    y_test = y_ref + np.random.randn(100).astype(np.float32) * noise
    ae = absolute_error(y_ref, y_test)
    re = relative_error(y_ref, y_test)
    cs = cosine_similarity(y_ref, y_test)
    snr = snr_db(y_ref, y_test)
    ac = allclose_check(y_ref, y_test)
    print(f"{noise:10.1e}  {np.max(ae):10.2e}  {np.max(re):10.2e}  {cs:12.9f}  {snr:10.2f}  {str(ac):>8s}")

<a id="6"></a>
## Exercise 6 — Numerical Validation Against ORT

Build an ONNX model, run it with ONNX Runtime, and compare outputs to a NumPy reference.

In [ ]:
# Build linear model Y = relu(X @ W + B)
np.random.seed(42)
W_np = np.random.randn(784, 256).astype(np.float32)
B_np = np.random.randn(256).astype(np.float32)

X_info = helper.make_tensor_value_info("X", TensorProto.FLOAT, [None, 784])
Y_info = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
W = helper.make_tensor("W", TensorProto.FLOAT, [784, 256], W_np.flatten())
B = helper.make_tensor("B", TensorProto.FLOAT, [256], B_np.flatten())

graph = helper.make_graph(
    [
        helper.make_node("MatMul", ["X", "W"], ["XW"]),
        helper.make_node("Add", ["XW", "B"], ["H"]),
        helper.make_node("Relu", ["H"], ["Y"]),
    ],
    "relu_linear", [X_info], [Y_info], initializer=[W, B],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
checker.check_model(model)

try:
    sess = ort.InferenceSession(model.SerializeToString())

    print(f"{'Test':>5s}  {'Batch':>5s}  {'Max Abs':>10s}  {'Cosine':>12s}  {'SNR (dB)':>10s}  {'Pass':>5s}")
    print("-" * 55)

    for i in range(8):
        batch = np.random.choice([1, 4, 8, 16, 32])
        x_np = np.random.randn(batch, 784).astype(np.float32)

        y_ref = np.maximum(0, x_np @ W_np + B_np)
        y_ort = sess.run(None, {"X": x_np})[0]

        ae = np.max(absolute_error(y_ref, y_ort))
        cs = cosine_similarity(y_ref, y_ort)
        snr = snr_db(y_ref, y_ort)
        ok = allclose_check(y_ref, y_ort)

        print(f"{i+1:5d}  {batch:5d}  {ae:10.2e}  {cs:12.9f}  {snr:10.2f}  {'OK' if ok else 'FAIL':>5s}")

except ImportError:
    print("onnxruntime not installed — skipping ORT validation.")

<a id="7"></a>
## Exercise 7 — Error Distribution Analysis

Visualize the distribution of absolute errors across output elements.

In [ ]:
import matplotlib.pyplot as plt

try:
    x_np = np.random.randn(64, 784).astype(np.float32)
    y_ref = np.maximum(0, x_np @ W_np + B_np)
    y_ort = sess.run(None, {"X": x_np})[0]

    abs_err = absolute_error(y_ref, y_ort).flatten()
    rel_err = relative_error(y_ref, y_ort).flatten()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Absolute error histogram
    axes[0].hist(abs_err, bins=50, color="steelblue", alpha=0.8, edgecolor="white")
    axes[0].axvline(np.mean(abs_err), color="red", linestyle="--",
                    label=f"mean={np.mean(abs_err):.2e}")
    axes[0].set_xlabel("Absolute Error")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Absolute Error Distribution")
    axes[0].legend(fontsize=8)

    # Relative error histogram
    axes[1].hist(rel_err[rel_err < np.percentile(rel_err, 99)], bins=50,
                 color="coral", alpha=0.8, edgecolor="white")
    axes[1].set_xlabel("Relative Error")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Relative Error Distribution (99th pct)")

    # Scatter: ref vs test
    subsample = np.random.choice(len(y_ref.flatten()), min(1000, len(y_ref.flatten())), replace=False)
    axes[2].scatter(y_ref.flatten()[subsample], y_ort.flatten()[subsample],
                    alpha=0.3, s=5, color="steelblue")
    lims = [min(y_ref.min(), y_ort.min()), max(y_ref.max(), y_ort.max())]
    axes[2].plot(lims, lims, 'r--', linewidth=1, label="y=x")
    axes[2].set_xlabel("Reference")
    axes[2].set_ylabel("ORT Output")
    axes[2].set_title("Reference vs ORT")
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

except NameError:
    print("onnxruntime session not available — skipping visualization.")

<a id="8"></a>
## Exercise 8 — Build a Validation Harness

Create a reusable validation function that runs structural, type, and numerical checks.
This harness can be integrated into CI pipelines.

In [ ]:
class ONNXValidator:
    """Reusable ONNX model validation harness."""

    def __init__(self, atol=1e-5, rtol=1e-4, min_snr_db=30.0, min_cosine=0.9999):
        self.atol = atol
        self.rtol = rtol
        self.min_snr_db = min_snr_db
        self.min_cosine = min_cosine

    def check_structural(self, model):
        try:
            checker.check_model(model)
            return True, "OK"
        except Exception as e:
            return False, str(e)[:200]

    def check_types(self, model):
        try:
            shape_inference.infer_shapes(model, check_type=True)
            return True, "OK"
        except Exception as e:
            return False, str(e)[:200]

    def check_numerical(self, model, reference_fn, test_inputs, input_name="X"):
        try:
            sess = ort.InferenceSession(model.SerializeToString())
        except Exception as e:
            return False, f"Session creation failed: {e}", []

        metrics_list = []
        all_pass = True

        for x in test_inputs:
            y_ref = reference_fn(x)
            y_ort = sess.run(None, {input_name: x})[0]

            m = {
                "max_abs_error": float(np.max(absolute_error(y_ref, y_ort))),
                "cosine_sim": float(cosine_similarity(y_ref, y_ort)),
                "snr_db": float(snr_db(y_ref, y_ort)),
                "allclose": bool(allclose_check(y_ref, y_ort, self.atol, self.rtol)),
            }

            passed = (m["allclose"] and
                      m["snr_db"] >= self.min_snr_db and
                      m["cosine_sim"] >= self.min_cosine)
            m["passed"] = passed
            all_pass = all_pass and passed
            metrics_list.append(m)

        return all_pass, "OK" if all_pass else "Numerical mismatch", metrics_list

    def validate(self, model, reference_fn=None, test_inputs=None, input_name="X"):
        report = {}

        ok, msg = self.check_structural(model)
        report["structural"] = {"passed": ok, "message": msg}
        if not ok:
            return report

        ok, msg = self.check_types(model)
        report["type_check"] = {"passed": ok, "message": msg}

        if reference_fn and test_inputs:
            ok, msg, metrics = self.check_numerical(model, reference_fn, test_inputs, input_name)
            report["numerical"] = {"passed": ok, "message": msg, "metrics": metrics}

        return report

# Run the validator
validator = ONNXValidator(atol=1e-5, rtol=1e-4)

ref_fn = lambda x: np.maximum(0, x @ W_np + B_np)
inputs = [np.random.randn(8, 784).astype(np.float32) for _ in range(10)]

report = validator.validate(model, ref_fn, inputs)

print("Validation Report")
print("=" * 40)
for check, result in report.items():
    status = "PASS" if result["passed"] else "FAIL"
    print(f"  {check:<15s}: {status}")
    if check == "numerical" and "metrics" in result:
        avg_snr = np.mean([m["snr_db"] for m in result["metrics"]])
        avg_cos = np.mean([m["cosine_sim"] for m in result["metrics"]])
        print(f"    Avg SNR: {avg_snr:.1f} dB, Avg Cosine: {avg_cos:.9f}")

<a id="9"></a>
## Exercise 9 — Before/After Optimization Equivalence

Apply graph optimizations and verify that the optimized model produces the same outputs.

In [ ]:
# Build a model with constant-foldable subgraph
np.random.seed(42)
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [None, 100])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

# Constants that can be folded: scale = a * b
a_val = np.array([2.0], dtype=np.float32)
b_val = np.array([0.5], dtype=np.float32)
W_val = np.random.randn(100, 10).astype(np.float32)

a_const = helper.make_node("Constant", [], ["a"],
    value=helper.make_tensor("a", TensorProto.FLOAT, [1], a_val))
b_const = helper.make_node("Constant", [], ["b"],
    value=helper.make_tensor("b", TensorProto.FLOAT, [1], b_val))
W_init = helper.make_tensor("W", TensorProto.FLOAT, [100, 10], W_val.flatten())

nodes = [
    a_const, b_const,
    helper.make_node("Mul", ["a", "b"], ["scale"]),
    helper.make_node("MatMul", ["X", "W"], ["XW"]),
    helper.make_node("Mul", ["XW", "scale"], ["Y"]),
]

graph = helper.make_graph(nodes, "foldable", [X], [Y], initializer=[W_init])
original_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
checker.check_model(original_model)

print(f"Original model: {len(original_model.graph.node)} nodes")
for n in original_model.graph.node:
    print(f"  {n.op_type}: {list(n.input)} -> {list(n.output)}")

# Apply constant folding via ORT
try:
    sess_orig = ort.InferenceSession(original_model.SerializeToString())

    # ORT applies optimizations internally — compare outputs
    test_inputs = [np.random.randn(4, 100).astype(np.float32) for _ in range(10)]

    ref_fn = lambda x: (x @ W_val) * (a_val * b_val)

    print("\nComparing original execution vs NumPy reference:")
    all_pass = True
    for i, x in enumerate(test_inputs):
        y_ref = ref_fn(x)
        y_ort = sess_orig.run(None, {"X": x})[0]
        cs = cosine_similarity(y_ref, y_ort)
        snr = snr_db(y_ref, y_ort)
        ok = allclose_check(y_ref, y_ort)
        all_pass = all_pass and ok
        if i < 3:
            print(f"  Test {i+1}: cos={cs:.9f}  SNR={snr:.1f}dB  pass={ok}")

    print(f"  ... ({len(test_inputs)} total tests)")
    print(f"  All passed: {all_pass}")

except ImportError:
    print("onnxruntime not installed — skipping optimization equivalence test.")

<a id="10"></a>
## Exercise 10 — Edge Case Testing

Test the model with edge-case inputs to verify robustness:
- All-zeros input
- Very large values
- Very small (subnormal) values
- Negative values

In [ ]:
edge_cases = {
    "zeros":      np.zeros((4, 784), dtype=np.float32),
    "ones":       np.ones((4, 784), dtype=np.float32),
    "large":      np.full((4, 784), 1e4, dtype=np.float32),
    "small":      np.full((4, 784), 1e-6, dtype=np.float32),
    "negative":   np.full((4, 784), -1.0, dtype=np.float32),
    "mixed":      np.random.choice([-1, 0, 1], size=(4, 784)).astype(np.float32),
}

# Use the linear model from Exercise 6
ref_fn = lambda x: np.maximum(0, x @ W_np + B_np)

try:
    sess = ort.InferenceSession(model.SerializeToString())

    print(f"{'Case':<12s}  {'Max Abs Err':>12s}  {'Cosine Sim':>12s}  {'SNR (dB)':>10s}  {'Has NaN':>7s}  {'Pass':>5s}")
    print("-" * 66)

    for name, x_np in edge_cases.items():
        y_ref = ref_fn(x_np)
        y_ort = sess.run(None, {"X": x_np})[0]

        has_nan = np.any(np.isnan(y_ort))
        if has_nan:
            print(f"{name:<12s}  {'NaN':>12s}  {'NaN':>12s}  {'NaN':>10s}  {'YES':>7s}  {'FAIL':>5s}")
            continue

        ae = np.max(absolute_error(y_ref, y_ort))
        cs = cosine_similarity(y_ref, y_ort)
        snr_val = snr_db(y_ref, y_ort)
        ok = allclose_check(y_ref, y_ort)
        print(f"{name:<12s}  {ae:12.2e}  {cs:12.9f}  {snr_val:10.2f}  {'NO':>7s}  {'OK' if ok else 'FAIL':>5s}")

except NameError:
    print("onnxruntime session not available — skipping edge case tests.")

In [ ]:
import matplotlib.pyplot as plt

# Visualize SNR across multiple validation runs
try:
    np.random.seed(0)
    snr_values = []
    cos_values = []

    for _ in range(50):
        x = np.random.randn(16, 784).astype(np.float32)
        y_ref = ref_fn(x)
        y_ort = sess.run(None, {"X": x})[0]
        snr_values.append(snr_db(y_ref, y_ort))
        cos_values.append(cosine_similarity(y_ref, y_ort))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.bar(range(len(snr_values)), snr_values, color="steelblue", alpha=0.7)
    ax1.axhline(30, color="red", linestyle="--", label="30 dB threshold")
    ax1.set_xlabel("Test Run")
    ax1.set_ylabel("SNR (dB)")
    ax1.set_title("SNR Across Validation Runs")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(range(len(cos_values)), cos_values, 'g-o', markersize=3)
    ax2.axhline(0.9999, color="red", linestyle="--", label="0.9999 threshold")
    ax2.set_xlabel("Test Run")
    ax2.set_ylabel("Cosine Similarity")
    ax2.set_title("Cosine Similarity Across Runs")
    ax2.legend()
    ax2.grid(alpha=0.3)
    ax2.set_ylim([min(0.9998, min(cos_values) - 0.0001), 1.0001])

    plt.tight_layout()
    plt.show()

except NameError:
    print("onnxruntime session not available — skipping visualization.")

In [ ]:
# Final summary: validation checklist
checklist = [
    ("Structural",  "onnx.checker.check_model()",        "Catches malformed graphs"),
    ("Type check",  "shape_inference(strict_mode=True)",  "Shape/dtype consistency"),
    ("Numerical",   "np.allclose(ref, ort, atol, rtol)",  "Element-wise correctness"),
    ("Cosine sim",  "dot(y,y_hat)/(||y||*||y_hat||)",     "Directional agreement"),
    ("SNR",         "10*log10(signal/noise) dB",          "Noise level assessment"),
    ("Edge cases",  "Zeros, large, small, negative",      "Robustness testing"),
    ("Regression",  "Before vs after optimization",       "Equivalence guarantee"),
    ("CI harness",  "Automated pipeline integration",     "Continuous validation"),
]

print(f"{'Check':<12s}  {'Method':<38s}  {'Purpose'}")
print("=" * 75)
for check, method, purpose in checklist:
    print(f"{check:<12s}  {method:<38s}  {purpose}")

## Summary

In these exercises you:

1. Used `onnx.checker.check_model()` to validate model structure and caught common errors.
2. Ran strict shape inference as a type-checking pass.
3. Implemented error metrics from scratch: absolute/relative error, cosine similarity, and SNR.
4. Performed numerical validation comparing ORT outputs to NumPy reference across multiple random inputs.
5. Visualized error distributions with histograms and scatter plots.
6. Built a reusable `ONNXValidator` class suitable for CI integration.
7. Tested model equivalence before and after graph optimization.
8. Verified robustness with edge-case inputs (zeros, large values, negatives).